# 02 CoM and Bounding Box

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`EXP004_KNN_Normal_CoM.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP004_KNN_Normal_CoM.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)
- [`EXP006_KNN_Normal_BB_Size_fixed.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP006_KNN_Normal_BB_Size_fixed.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/KNN/Normal/scripts/EXP004_KNN_Normal_CoM.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'ef712d43e820d69d6ec441722adc806a2032b36c'}, '05_model_optimize/KNN/Normal/scripts/EXP006_KNN_Normal_BB_Size_fixed.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'cee75c49e9c758d74a97e3441a06385c852d19ba'}}

## EXP004_KNN_Normal_CoM

원본: [05_model_optimize/KNN/Normal/scripts/EXP004_KNN_Normal_CoM.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP004_KNN_Normal_CoM.py)  
Git blob: `ef712d43e820d69d6ec441722adc806a2032b36c`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

EXP-004: KNN Normal CoM centering vs unchanged baseline.

Uses identical CoM centering as the existing MLP Normal EXP유004 code:
threshold 32/255, weighted mass, target (13.5,13.5), integer shift, zero-fill.
No original Test data, final Test data, or Hand-made images are used for fitting.



In [ ]:
%%archive_source 05_model_optimize/KNN/Normal/scripts/EXP004_KNN_Normal_CoM.py
#!/usr/bin/env python3
"""EXP-004: KNN Normal CoM centering vs unchanged baseline.

Uses identical CoM centering as the existing MLP Normal EXP유004 code:
threshold 32/255, weighted mass, target (13.5,13.5), integer shift, zero-fill.
No original Test data, final Test data, or Hand-made images are used for fitting.
"""
from __future__ import annotations

import csv
import hashlib
import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

ROOT = Path(__file__).resolve().parents[4]
sys.path.insert(0, str(ROOT / '03_experiments' / 'model_optimization'))
sys.path.insert(0, str(ROOT / '05_model_optimize' / 'MLP' / 'Normal' / 'scripts'))

from common import LABELS, fixed_split, pipeline, scores
from knn import build_baseline
from Automatic_Centering_Transformer import center_batch, test_centering

OUT = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'results' / 'EXP004_com_centering'
HANDMADE = ROOT / '02_data' / 'Hand_made'
EXPECTED_HANDMADE_SHA = {
    41: '0a829172a72787c770139c73de34cc8b751181829ff6fe08ec260247b9be5a2f',
    42: 'f364fb453d05b4fe2a705c2b8a80683b404608c7c257d664e3ec7ac86932d4c5',
    43: '10db20997c727c1bc1f1299f8e89626fadd3ec9618468d6759c86b0d2ddc4a66',
    44: '0e266a36e7b29edfcca0514836b5c383d13939948b4e335d70f61bb40bdc084d',
}

class BatchedCoM(BaseEstimator, TransformerMixin):
    """Memory-bounded wrapper of the repo's existing MLP CoM implementation."""
    def __init__(self, batch_size=2048):
        self.batch_size = batch_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        a = np.asarray(X, dtype=np.float32)
        if a.ndim != 2 or a.shape[1] != 784:
            raise ValueError(f'Expected (N,784) images, got {a.shape}')
        out = np.empty_like(a)
        for start in range(0, len(a), self.batch_size):
            stop = min(start + self.batch_size, len(a))
            out[start:stop] = center_batch(a[start:stop], 'mass')
        return out


def write_rows(path, rows, fields=None):
    if fields is None:
        fields = list(rows[0]) if rows else []
    with path.open('w', encoding='utf-8-sig', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)


def write_cm(path, cm):
    write_rows(path, [
        {'true/pred': LABELS[i], **{LABELS[j]: int(cm[i][j]) for j in range(16)}}
        for i in range(16)
    ])


def read_handmade():
    arrays, labs, writers, local_indices = [], [], [], []
    for w in (41, 42, 43, 44):
        path = HANDMADE / f'ml26_{w}.npz'
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest != EXPECTED_HANDMADE_SHA[w]:
            raise ValueError(f'Unexpected Hand-made SHA256 for writer {w}: {digest}')
        with np.load(path, allow_pickle=False) as d:
            if set(d.files) != {'img', 'label', 'label_id'}:
                raise ValueError(f'Unexpected Hand-made keys {d.files}')
            img = d['img']
            y = d['label_id'].astype(np.int64)
            label = d['label']
        if img.shape != (640, 28, 28) or img.dtype != np.uint8 or y.shape != (640,):
            raise ValueError(f'Unexpected images/labels for writer {w}')
        if not np.array_equal(y, np.array([LABELS.index(str(c)) for c in label], dtype=np.int64)):
            raise ValueError(f'Unexpected label mapping for writer {w}')
        if not np.array_equal(np.bincount(y, minlength=16), np.full(16, 40)):
            raise ValueError(f'Unexpected label counts for writer {w}')
        arrays.append(img.reshape(640, 784).astype(np.float32))
        labs.append(y)
        writers.extend([w] * 640)
        local_indices.extend(range(640))
    return np.concatenate(arrays), np.concatenate(labs), np.array(writers), np.array(local_indices)


def paired_stats(y, original, centered):
    old_correct = original == y
    new_correct = centered == y
    return {
        'corrected': int(np.sum(~old_correct & new_correct)),
        'new_errors': int(np.sum(old_correct & ~new_correct)),
        'unchanged_correct': int(np.sum(old_correct & new_correct)),
        'unchanged_wrong': int(np.sum(~old_correct & ~new_correct)),
    }


def main():
    test_centering()
    xtr, ytr, xva, yva, sha = fixed_split()
    xhm, yhm, writers, local_indices = read_handmade()
    print(f'Official Training/Validation: {len(ytr)}/{len(yva)}; Hand-made: {len(yhm)}', flush=True)
    raw_model, scaled = build_baseline()
    if not scaled:
        raise ValueError('KNN baseline is expected to use PixelScaler255')
    raw_pipe = pipeline(raw_model, scaled)
    candidate_model, _ = build_baseline()
    candidate_scaled_pipe = pipeline(candidate_model, True)
    centered_pipe = Pipeline([('center_of_mass', BatchedCoM()), *candidate_scaled_pipe.steps])

    print('Fitting unchanged KNN baseline...', flush=True)
    t = time.perf_counter()
    raw_pipe.fit(xtr, ytr)
    raw_fit = time.perf_counter() - t
    pred_raw_va = raw_pipe.predict(xva)
    pred_raw_hm = raw_pipe.predict(xhm)
    baseline_va = scores(yva, pred_raw_va)
    baseline_hm = scores(yhm, pred_raw_hm)
    print(f"Baseline Official={baseline_va['accuracy']:.6f}, Hand-made={baseline_hm['accuracy']:.6f}", flush=True)
    if (not np.isclose(baseline_va['accuracy'], 0.9629166666666666, atol=1e-12)
        or not np.isclose(baseline_hm['accuracy'], 0.58828125, atol=1e-12)
        or not np.isclose(baseline_hm['macro_f1'], 0.5664200025438473, atol=1e-12)):
        raise ValueError('EXP-001/002 baseline results were not reproduced; refusing comparison')

    print('Fitting KNN with CoM on official Training only...', flush=True)
    t = time.perf_counter()
    centered_pipe.fit(xtr, ytr)
    com_fit = time.perf_counter() - t
    pred_com_va = centered_pipe.predict(xva)
    pred_com_hm = centered_pipe.predict(xhm)
    com_va = scores(yva, pred_com_va)
    com_hm = scores(yhm, pred_com_hm)
    print(f"CoM Official={com_va['accuracy']:.6f}, MacroF1={com_va['macro_f1']:.6f}", flush=True)
    print(f"CoM Hand-made={com_hm['accuracy']:.6f}, MacroF1={com_hm['macro_f1']:.6f}", flush=True)
    print(f"Difference Official={100*(com_va['accuracy']-baseline_va['accuracy']):+.3f} percentage points", flush=True)
    print(f"Difference Hand-made={100*(com_hm['accuracy']-baseline_hm['accuracy']):+.3f} percentage points", flush=True)

    OUTPUT = OUT
    OUTPUT.mkdir(parents=True, exist_ok=True)
    write_cm(OUTPUT / 'official_com_confusion_matrix.csv', com_va['confusion_matrix'])
    write_cm(OUTPUT / 'handmade_com_confusion_matrix.csv', com_hm['confusion_matrix'])

    summary_rows = []
    for name, base, com in [('official_validation', baseline_va, com_va), ('handmade_dev', baseline_hm, com_hm)]:
        summary_rows.append({'evaluation': name, 'n': len(yva) if name=='official_validation' else len(yhm),
                             'baseline_accuracy': base['accuracy'], 'com_accuracy': com['accuracy'],
                             'delta_accuracy_pp': 100*(com['accuracy']-base['accuracy']),
                             'baseline_macro_f1': base['macro_f1'], 'com_macro_f1': com['macro_f1'],
                             'delta_macro_f1': com['macro_f1']-base['macro_f1']})
    write_rows(OUTPUT / 'comparison.csv', summary_rows)

    writer_rows = []
    for w in (41,42,43,44):
        mask = writers == w
        old, new = scores(yhm[mask], pred_raw_hm[mask]), scores(yhm[mask], pred_com_hm[mask])
        writer_rows.append({'writer': w, 'n': int(mask.sum()), 'baseline_accuracy': old['accuracy'],
                            'com_accuracy': new['accuracy'], 'delta_pp': 100*(new['accuracy']-old['accuracy']),
                            'baseline_macro_f1': old['macro_f1'], 'com_macro_f1': new['macro_f1'],
                            **paired_stats(yhm[mask], pred_raw_hm[mask], pred_com_hm[mask])})
        print(f"Writer {w}: {old['accuracy']:.4f} -> {new['accuracy']:.4f}", flush=True)
    write_rows(OUTPUT / 'by_writer.csv', writer_rows)

    pc_rows = []
    for i, c in enumerate(LABELS):
        b, n = baseline_hm['per_class'][c], com_hm['per_class'][c]
        pc_rows.append({'class': c, 'n': b['support'], 'baseline_recall': b['recall'],
                        'com_recall': n['recall'], 'delta_recall_pp': 100*(n['recall']-b['recall']),
                        'baseline_f1': b['f1'], 'com_f1': n['f1']})
    write_rows(OUTPUT / 'per_class_handmade.csv', pc_rows)

    pred_rows = []
    for w, idx, y, a, b in zip(writers, local_indices, yhm, pred_raw_hm, pred_com_hm):
        pred_rows.append({'writer': int(w), 'index': int(idx), 'true': LABELS[int(y)],
                          'baseline_prediction': LABELS[int(a)], 'com_prediction': LABELS[int(b)],
                          'baseline_correct': int(a==y), 'com_correct': int(b==y)})
    write_rows(OUTPUT / 'handmade_predictions_comparison.csv', pred_rows)

    result = {
        'experiment':'EXP-004 KNN Normal CoM centering',
        'source_centering':'05_model_optimize/MLP/Normal/scripts/Automatic_Centering_Transformer.py',
        'center_method':'mass', 'threshold_raw':32, 'target_xy':[13.5,13.5],
        'shift':'np.rint integer translation, zero fill; no wrap/no interpolation',
        'training_n':len(ytr),'validation_n':len(yva),'handmade_n':len(yhm),
        'official_dataset_sha256':sha, 'handmade_sha256':{str(k):v for k,v in EXPECTED_HANDMADE_SHA.items()},
        'model_hyperparameters': {str(k):repr(v) for k,v in raw_model.get_params().items()},
        'python':platform.python_version(),'numpy':np.__version__,'sklearn':sklearn.__version__,
        'fit_seconds':{'baseline':raw_fit, 'com':com_fit},
        'official_validation':{'baseline':baseline_va,'com':com_va,'paired':paired_stats(yva,pred_raw_va,pred_com_va)},
        'handmade_dev':{'baseline':baseline_hm,'com':com_hm,'paired':paired_stats(yhm,pred_raw_hm,pred_com_hm)},
        'by_writer':writer_rows,
        'note':'CoM is fixed from MLP project parameters; no Hand-made labels used to fit/tune transforms. Original Test and final Test unused.'
    }
    (OUTPUT / 'metrics.json').write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding='utf-8')
    print('Paired official:', result['official_validation']['paired'], flush=True)
    print('Paired Hand-made:', result['handmade_dev']['paired'], flush=True)
    print('Saved:', OUTPUT, flush=True)

if __name__ == '__main__':
    main()


## EXP006_KNN_Normal_BB_Size_fixed

원본: [05_model_optimize/KNN/Normal/scripts/EXP006_KNN_Normal_BB_Size_fixed.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP006_KNN_Normal_BB_Size_fixed.py)  
Git blob: `cee75c49e9c758d74a97e3441a06385c852d19ba`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

EXP-006: KNN Normal geometry study: CoM vs BB-size vs BB-size-then-CoM.

Place in 05_model_optimize/KNN/Normal/scripts/ alongside EXP004_KNN_Normal_CoM.py.
Uses official Training fold only for fitting, Validation and 4-writer Hand-made
for development assessment. Does not load Original Test or final Test.

Size transform follows the *exact geometry and scipy affine_transform formula*
from MLP/Combined/scripts/exp_cmb004_geometry_normalization.py and
MLP/Normal/scripts/exp006_size_com_comparison.



In [ ]:
%%archive_source 05_model_optimize/KNN/Normal/scripts/EXP006_KNN_Normal_BB_Size_fixed.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP-006: KNN Normal geometry study: CoM vs BB-size vs BB-size-then-CoM.

Place in 05_model_optimize/KNN/Normal/scripts/ alongside EXP004_KNN_Normal_CoM.py.
Uses official Training fold only for fitting, Validation and 4-writer Hand-made
for development assessment. Does not load Original Test or final Test.

Size transform follows the *exact geometry and scipy affine_transform formula*
from MLP/Combined/scripts/exp_cmb004_geometry_normalization.py and
MLP/Normal/scripts/exp006_size_com_comparison.py (threshold=32,
train-only median bbox long-side target, order=1, constant zero fill).
CoM uses the existing MLP Normal Automatic_Centering_Transformer.py via
EXP004_KNN_Normal_CoM.BatchedCoM.
"""
from __future__ import annotations

import csv
import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import scipy
import sklearn
from scipy.ndimage import affine_transform
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

ROOT = Path(__file__).resolve().parents[4]
sys.path.insert(0, str(ROOT / '03_experiments' / 'model_optimization'))
sys.path.insert(0, str(ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'))

from common import LABELS, fixed_split, pipeline, scores
from EXP004_KNN_Normal_CoM import BatchedCoM, read_handmade, paired_stats
from Automatic_Centering_Transformer import test_centering  # imported by EXP004

OUTPUT = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'results' / 'EXP006_bb_size'
EXPECTED_COM_OFFICIAL_CORRECT = 18529  # EXP-005 k3_distance, N=19200
EXPECTED_COM_HANDMADE_CORRECT = 2017  # EXP-005 k3_distance, N=2560
GUARDRAIL_OFFICIAL = 0.9569791666666667  # EXP-005 selection guardrail (>=95.6979%)
THRESHOLD = 32
TARGET_COM = (13.5, 13.5)


def geometry(batch):
    """Same bbox long-side/center definition as MLP Combined EXP-CMB-004."""
    a = np.asarray(batch, dtype=np.float32).reshape(-1, 28, 28)
    fg = a >= THRESHOLD
    if not np.all(fg.any(axis=(1, 2))):
        raise ValueError('Empty foreground image at threshold 32: cannot size-normalize')
    xx = fg.any(axis=1)
    yy = fg.any(axis=2)
    xmin = xx.argmax(axis=1)
    xmax = 27 - xx[:, ::-1].argmax(axis=1)
    ymin = yy.argmax(axis=1)
    ymax = 27 - yy[:, ::-1].argmax(axis=1)
    width = xmax - xmin + 1
    height = ymax - ymin + 1
    return (np.maximum(width, height), (ymin + ymax) / 2.0, (xmin + xmax) / 2.0)


def apply_size(batch, target):
    """Exact BB-size affine rule used by the MLP Combined reference code."""
    a = np.asarray(batch, dtype=np.float32).reshape(-1, 28, 28)
    longs, cy, cx = geometry(a)
    factors = float(target) / longs.astype(float)
    out = np.empty_like(a)
    for i in range(len(a)):
        inv = 1.0 / factors[i]
        center = (cy[i], cx[i])
        img = affine_transform(a[i], np.eye(2) * inv,
              offset=(center[0] * (1 - inv), center[1] * (1 - inv)),
              output_shape=(28, 28), order=1,
              mode='constant', cval=0.0, prefilter=False)
        out[i] = np.clip(img, 0, 255)
    return out.reshape(-1, 784), factors


class BatchedBBSize:
    """A standalone fit/transform object; target is trained on official train only."""
    def __init__(self, target_longside, batch_size=2048):
        self.target_longside = int(target_longside)
        self.batch_size = int(batch_size)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        a = np.asarray(X, dtype=np.float32)
        if a.ndim != 2 or a.shape[1] != 784:
            raise ValueError(f'Expected (N,784), got {a.shape}')
        out = np.empty_like(a)
        for start in range(0, len(a), self.batch_size):
            stop = min(start + self.batch_size, len(a))
            out[start:stop], _ = apply_size(a[start:stop], self.target_longside)
        return out


def write_csv(path, rows):
    if not rows:
        raise ValueError('No rows to write')
    with path.open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def save_cm(path, matrix):
    write_csv(path, [
        {'true/pred': LABELS[i], **{LABELS[j]: int(matrix[i][j]) for j in range(16)}}
        for i in range(16)
    ])


def invariance_tests():
    test_centering()
    x = np.zeros((1, 28, 28), dtype=np.float32)
    x[0, 6:22, 9:19] = 255.0
    # Identity when target equals actual bbox long side
    y, factors = apply_size(x, 16)
    if not (np.allclose(y, x.reshape(1, 784)) and np.allclose(factors, 1.0)):
        raise AssertionError('BB-size identity test failed')
    scaled, _ = apply_size(x, 20)
    if not (scaled.shape == (1, 784) and np.isfinite(scaled).all()
            and 0 <= scaled.min() <= scaled.max() <= 255):
        raise AssertionError('BB-size range/shape test failed')
    print('GEOMETRY SELF-TEST PASSED: label-free BB resizing and MLP CoM', flush=True)


def model_for(method, target):
    knn = KNeighborsClassifier(n_neighbors=3, weights='distance',
                               metric='minkowski', p=2)
    steps = []
    if method in ('bbox_size', 'size_then_com'):
        steps.append(('bbox_size', BatchedBBSize(target)))
    if method in ('com', 'size_then_com'):
        steps.append(('center_of_mass', BatchedCoM()))
    steps.extend(pipeline(knn, True).steps)
    return Pipeline(steps)


def score_rows(method, true, predictions):
    stats = scores(true, predictions)
    return {'method': method, 'n': len(true), 'accuracy': stats['accuracy'],
            'macro_f1': stats['macro_f1']}, stats


def main():
    invariance_tests()
    xtr, ytr, xva, yva, dataset_sha = fixed_split()
    xhm, yhm, writers, local_indices = read_handmade()
    print(f'Data: official train={len(ytr)}, validation={len(yva)}, Hand-made={len(yhm)}', flush=True)
    target = int(np.median(geometry(xtr)[0]))
    print('BB target long side from OFFICIAL TRAIN ONLY:', target, 'pixels', flush=True)
    if target != 20:
        raise AssertionError(f'Expected MLP reference target 20, got {target}. Check official split.')

    methods = ('com', 'bbox_size', 'size_then_com')
    pred_va_all, pred_hm_all = {}, {}
    results, writer_rows, class_rows, prediction_rows = [], [], [], []
    fit_times, predict_times = {}, {}
    scored = {}

    for method in methods:
        print(f'\n[{method}] Fitting official Training ONLY...', flush=True)
        model = model_for(method, target)
        begin = time.perf_counter()
        model.fit(xtr, ytr)
        fit_times[method] = time.perf_counter() - begin

        begin = time.perf_counter()
        pred_va = model.predict(xva).astype(np.int64)
        pred_hm = model.predict(xhm).astype(np.int64)
        predict_times[method] = time.perf_counter() - begin

        pred_va_all[method], pred_hm_all[method] = pred_va, pred_hm
        ovrow, ovs = score_rows(method, yva, pred_va)
        hmrow, hms = score_rows(method, yhm, pred_hm)
        scored[method] = {'official': ovs, 'handmade': hms}
        results.append({'method': method,
                        'official_accuracy': ovs['accuracy'], 'official_macro_f1': ovs['macro_f1'],
                        'handmade_accuracy': hms['accuracy'], 'handmade_macro_f1': hms['macro_f1'],
                        'guardrail_pass': int(ovs['accuracy'] >= GUARDRAIL_OFFICIAL)})
        print(f"{method:14s} Official={ovs['accuracy']:.6f} F1={ovs['macro_f1']:.6f} | "
              f"Hand-made={hms['accuracy']:.6f} F1={hms['macro_f1']:.6f} | "
              f"guardrail={'PASS' if ovs['accuracy'] >= GUARDRAIL_OFFICIAL else 'FAIL'}", flush=True)
        for writer in (41, 42, 43, 44):
            mask = writers == writer
            s = scores(yhm[mask], pred_hm[mask])
            writer_rows.append({'method': method, 'writer': writer, 'n': int(mask.sum()),
                                'accuracy': s['accuracy'], 'macro_f1': s['macro_f1']})
        for c in LABELS:
            s = hms['per_class'][c]
            class_rows.append({'method': method, 'class': c, 'n': s['support'],
                               'precision': s['precision'], 'recall': s['recall'], 'f1': s['f1']})

    if (int(np.sum(pred_va_all['com'] == yva)) != EXPECTED_COM_OFFICIAL_CORRECT
        or int(np.sum(pred_hm_all['com'] == yhm)) != EXPECTED_COM_HANDMADE_CORRECT):
        raise RuntimeError('EXP-005 CoM k3_distance reference did NOT reproduce. '
                           'Check environment/split/preprocessing before comparing results.')
    print('\nEXP-005 k3_distance CoM reference verified exactly (correct counts).', flush=True)
    for row in results:
        row['official_delta_pp_vs_com'] = 100 * (row['official_accuracy'] - results[0]['official_accuracy'])
        row['handmade_delta_pp_vs_com'] = 100 * (row['handmade_accuracy'] - results[0]['handmade_accuracy'])

    chosen_pool = [row for row in results if row['guardrail_pass']]
    chosen = max(chosen_pool, key=lambda row: (row['handmade_macro_f1'], row['handmade_accuracy'],
                                              row['official_macro_f1'])) if chosen_pool else None

    OUTPUT.mkdir(parents=True, exist_ok=True)
    write_csv(OUTPUT / 'candidate_results.csv', results)
    write_csv(OUTPUT / 'by_writer.csv', writer_rows)
    write_csv(OUTPUT / 'per_class_handmade.csv', class_rows)

    for method in methods:
        save_cm(OUTPUT / f'handmade_cm_{method}.csv', scored[method]['handmade']['confusion_matrix'])
        save_cm(OUTPUT / f'official_cm_{method}.csv', scored[method]['official']['confusion_matrix'])
        for i in range(len(yhm)):
            prediction_rows.append({'method': method, 'writer': int(writers[i]),
                                    'index': int(local_indices[i]), 'true': LABELS[int(yhm[i])],
                                    'predicted': LABELS[int(pred_hm_all[method][i])],
                                    'correct': int(pred_hm_all[method][i] == yhm[i])})
    write_csv(OUTPUT / 'handmade_predictions.csv', prediction_rows)
    transitions = {
        name: {
            'official': paired_stats(yva, pred_va_all['com'], pred_va_all[name]),
            'handmade': paired_stats(yhm, pred_hm_all['com'], pred_hm_all[name])
        } for name in methods if name != 'com'
    }
    report = {
        'experiment': 'EXP-006 KNN Normal BB-size vs CoM vs BB-size-then-CoM',
        'methods': list(methods),
        'reference': 'EXP-005 CoM + k3 distance',
        'training_n': int(len(ytr)), 'validation_n': int(len(yva)), 'handmade_n': int(len(yhm)),
        'official_dataset_sha256': dataset_sha,
        'target_bbox_longside': target, 'target_fit': 'median official Training bbox long side only',
        'bbox_threshold': THRESHOLD,
        'bbox_interpolation': 'scipy.ndimage.affine_transform order=1, constant zero fill, centered on bbox',
        'com_method': 'MLP Normal mass center, threshold 32, target (13.5,13.5), integer zero-filled shifts',
        'model': 'KNeighborsClassifier(n_neighbors=3,weights=distance,p=2)',
        'official_accuracy_guardrail': GUARDRAIL_OFFICIAL,
        'selection': 'Among guardrail passes: highest Hand-made Macro F1, then accuracy, official macro F1',
        'candidates': results, 'selected_development_candidate': chosen,
        'paired_vs_com': transitions,
        'fit_seconds': fit_times, 'predict_seconds': predict_times,
        'python': platform.python_version(), 'numpy': np.__version__,
        'scipy': scipy.__version__, 'sklearn': sklearn.__version__,
        'limitations': 'Hand-made reused for development; Original Test and final professor Test untouched. '
                       'BB-size method is NOT BB-center: it changes scale around bbox center. '
                       'No augmentation. No independent final score.'
    }
    (OUTPUT / 'metrics.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    print('\nPAIRED vs CoM:')
    for name, pair in transitions.items():
        print(name, pair)
    if chosen:
        print('\nSELECTED DEVELOPMENT CANDIDATE:', chosen['method'])
        print(f"Official={chosen['official_accuracy']:.6f}, "
              f"Hand-made={chosen['handmade_accuracy']:.6f}, "
              f"MacroF1={chosen['handmade_macro_f1']:.6f}")
    else:
        print('No candidates pass official Validation guardrail.')
    print('Saved:', OUTPUT)


if __name__ == '__main__':
    main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")